<a href="https://colab.research.google.com/github/ed1000s0n/Python_projects/blob/main/Projeto_final_sistemas_de_controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip install control==0.9.4 plotly "nbformat>=4.2.0" tqdm
import control as ct
import plotly.graph_objects as go

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.1/455.1 kB 7.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
from tqdm import tqdm

In [ ]:
zeta1 = 1.0
t_acomodacao1 = 10

T = np.arange(0, 15, 0.005)

best_k = None
min_error = float("inf")
target_value = 10
step = 0.0001

k_values = np.arange(4, 6 + step, step)

for k in tqdm(k_values, desc="Procurando melhor valor de k"):
    omega_n1 = k / (t_acomodacao1 * zeta1)
    num1 = [omega_n1**2]
    den1 = [1, 2 * zeta1 * omega_n1, omega_n1**2]
    G1 = ct.tf(num1, den1)

    t1, y1 = ct.step_response(G1, T)

    indices_98 = np.where(y1 >= 0.98)[0]

    if len(indices_98) > 0:
        t_acomodacao_atual = t1[indices_98[0]]

        error = abs(t_acomodacao_atual - target_value)

        if error < min_error:
            min_error = error
            best_k = k
            best_t_acomodacao = t_acomodacao_atual
            best_y_at_acomodacao = y1[indices_98[0]]

print(f"Melhor valor de k encontrado: {best_k:.8f}")
print(f"Tempo de acomodação de 2% (t1 quando y1 >= 0.98): {best_t_acomodacao:.6f} s")
print(f"Valor de y1 no tempo de acomodação: {best_y_at_acomodacao:.6f}")
print(f"Erro em relação a 10s: {min_error:.6f}")

omega_n1 = best_k / (t_acomodacao1 * zeta1)

num1 = [omega_n1**2]
den1 = [1, 2 * zeta1 * omega_n1, omega_n1**2]
G1 = ct.tf(
    num1,
    den1,
)

Procurando melhor valor de k: 100%|██████████| 20001/20001 [10:23<00:00, 32.08it/s]

Melhor valor de k encontrado: 5.83400000
Tempo de acomodação de 2% (t1 quando y1 >= 0.98): 10.000000 s
Valor de y1 no tempo de acomodação: 0.980001
Erro em relação a 10s: 0.000000


In [ ]:
Mp = 0.1
t_acomodacao2 = 10.0

zeta2 = -np.log(Mp) / np.sqrt(np.pi**2 + (np.log(Mp)) ** 2)
omega_n2 = 4 / (zeta2 * t_acomodacao2)
num2 = [omega_n2**2]
den2 = [1, 2 * zeta2 * omega_n2, omega_n2**2]
G2 = ct.tf(num2, den2)

In [ ]:
T = np.arange(0, 15, 0.005)
t1, y1 = ct.step_response(G1, T)
t2, y2 = ct.step_response(G2, T)

print("\n=== VERIFICAÇÃO DOS SISTEMAS ===")
print(f"\nG1(s) = {G1}")
print(f"G2(s) = {G2}")


=== VERIFICAÇÃO DOS SISTEMAS ===

G1(s) = 
        0.3404
----------------------
s^2 + 1.167 s + 0.3404

G2(s) = 
       0.4578
--------------------
s^2 + 0.8 s + 0.4578



In [ ]:
fig = go.Figure()

grafico_G1 = go.Scatter(
    x=t1,
    y=y1,
    name=f"Sistema 1: ζ={zeta1} (sem ultrapassagem)",
    mode="lines",
    line={"color": "blue", "width": 3},
)

grafico_G2 = go.Scatter(
    x=t2,
    y=y2,
    name=f"Sistema 2: ζ={zeta2:.3f} (10% ultrapassagem)",
    mode="lines",
    line={"color": "red", "width": 3},
)

linha_ts = go.Scatter(
    x=[10, 10],
    y=[0, 1.15],
    mode="lines",
    line={"color": "gray", "width": 2, "dash": "dash"},
    name="ts = 10s",
    showlegend=True,
)

linha_final = go.Scatter(
    x=[0, 15],
    y=[1.0, 1.0],
    mode="lines",
    line={"color": "green", "width": 2, "dash": "dot"},
    name="Amplitude final (1.0)",
    showlegend=True,
)

fig.add_trace(grafico_G1)
fig.add_trace(grafico_G2)
fig.add_trace(linha_ts)
fig.add_trace(linha_final)

fig.update_layout(
    title="Comparação: Resposta ao Degrau - Sistemas de 2ª Ordem (ts = 10s)",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude",
    separators=",.",
    template="seaborn",
)

fig.show()

In [ ]:
G_5 = ct.tf([1, 27, 150, 200], [1, 75.2, 1465, 10790, 27100, 5000])

print("=== Sistema de Quinta Ordem ===")
print(G_5)

=== Sistema de Quinta Ordem ===

                 s^3 + 27 s^2 + 150 s + 200
-------------------------------------------------------------
s^5 + 75.2 s^4 + 1465 s^3 + 1.079e+04 s^2 + 2.71e+04 s + 5000



In [ ]:
T_sim = np.arange(0, 20, 0.01)
t_5, y_5 = ct.step_response(G_5, T_sim)

ganho_estatico = ct.dcgain(G_5)
print(f"Ganho estático G(0) = {ganho_estatico:.4f}")
print(f"\nInterpretação: O valor final da resposta ao degrau é {ganho_estatico:.4f}")

fig = go.Figure()

grafico_G5 = go.Scatter(
    x=t_5,
    y=y_5,
    name="Resposta ao Degrau - Sistema de 5ª Ordem",
    mode="lines",
    line={"color": "blue", "width": 3},
)

linha_G0 = go.Scatter(
    x=[0, T_sim[-1]],
    y=[ganho_estatico, ganho_estatico],
    mode="lines",
    line={"color": "red", "width": 2, "dash": "dash"},
    name=f"G(0) = {ganho_estatico:.4f} (Ganho Estático)",
)

valor_final = go.Scatter(
    x=[T_sim[-1]],
    y=[ganho_estatico],
    mode="markers+text",
    marker={"size": 12, "color": "red", "symbol": "circle"},
    text=[f"G(0)={ganho_estatico:.4f}"],
    textposition="top right",
    name="",
    showlegend=False,
)

fig.add_trace(grafico_G5)
fig.add_trace(linha_G0)
fig.add_trace(valor_final)

fig.update_layout(
    title="Identificação Visual do Ganho Estático G(0)",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude",
    separators=",.",
    template="seaborn",
    legend={"yanchor": "top", "y": 0.99, "xanchor": "left", "x": 0.01},
)

fig.show()

Ganho estático G(0) = 0.0400

Interpretação: O valor final da resposta ao degrau é 0.0400


In [ ]:
valor_63 = 0.632 * ganho_estatico
indice_63 = np.argmin(np.abs(y_5 - valor_63))
tau_1 = t_5[indice_63]

print(f"Valor correspondente a 63.2% do ganho estático: {valor_63:.4f}")
print(f"Constante de tempo estimada (τ): {tau_1:.4f} s")

fig = go.Figure()

grafico_G5 = go.Scatter(
    x=t_5,
    y=y_5,
    name="Resposta ao Degrau - Sistema de 5ª Ordem",
    mode="lines",
    line={"color": "blue", "width": 3},
)

linha_63 = go.Scatter(
    x=[0, T_sim[-1]],
    y=[valor_63, valor_63],
    mode="lines",
    line={"color": "orange", "width": 2, "dash": "dot"},
    name=f"63.2% de G(0) = {valor_63:.4f}",
)

linha_tau = go.Scatter(
    x=[tau_1, tau_1],
    y=[0, ganho_estatico],
    mode="lines",
    line={"color": "green", "width": 2, "dash": "dash"},
    name=f"τ = {tau_1:.2f} s",
)

ponto_tau = go.Scatter(
    x=[tau_1],
    y=[valor_63],
    mode="markers+text",
    marker={"size": 15, "color": "red", "symbol": "circle"},
    text=[f"(τ={tau_1:.2f}s, {valor_63:.4f})"],
    textposition="top right",
    name="Ponto de 63.2%",
    showlegend=False,
)

linha_G0 = go.Scatter(
    x=[0, T_sim[-1]],
    y=[ganho_estatico, ganho_estatico],
    mode="lines",
    line={"color": "green", "width": 1.5, "dash": "dot"},
    name=f"G(0) = {ganho_estatico:.4f}",
)

fig.add_trace(grafico_G5)
fig.add_trace(linha_G0)
fig.add_trace(linha_63)
fig.add_trace(linha_tau)
fig.add_trace(ponto_tau)

fig.update_layout(
    title="Identificação Visual da Constante de Tempo τ",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude",
    separators=",.",
    template="seaborn",
    legend={"yanchor": "top", "y": 0.99, "xanchor": "left", "x": 0.01},
)

fig.show()

Valor correspondente a 63.2% do ganho estático: 0.0253
Constante de tempo estimada (τ): 4.6400 s


In [ ]:
G_1 = ct.tf([ganho_estatico], [tau_1, 1])

print(f"Ganho estático K = G(0) = {ganho_estatico:.4f}")
print(f"Constante de tempo τ = {tau_1:.4f} s")
print()

print("Sistema de Primeira Ordem Aproximado")
print(G_1)
print()

t_1, y_1 = ct.step_response(G_1, T_sim)

Ganho estático K = G(0) = 0.0400
Constante de tempo τ = 4.6400 s

=== Sistema de Primeira Ordem Aproximado ===

   0.04
----------
4.64 s + 1




In [ ]:
fig = go.Figure()

grafico_step_5 = go.Scatter(
    x=t_5,
    y=y_5,
    name="Sistema de 5ª Ordem (Original)",
    mode="lines",
    line={"color": "green", "width": 4},
)

grafico_step_1 = go.Scatter(
    x=t_1,
    y=y_1,
    name=f"Sistema de 1ª Ordem (Aproximado: K={ganho_estatico:.4f}, τ={tau_1:.2f}s)",
    mode="lines",
    line={"color": "red", "width": 3, "dash": "dash"},
)

linha_final = go.Scatter(
    x=[0, T_sim[-1]],
    y=[ganho_estatico, ganho_estatico],
    mode="lines",
    line={"color": "blue", "width": 2, "dash": "dot"},
    name=f"G(0) = {ganho_estatico:.4f}",
    showlegend=True,
)

fig.add_trace(grafico_step_5)
fig.add_trace(grafico_step_1)
fig.add_trace(linha_final)

fig.update_layout(
    title="Comparação: Resposta ao Degrau - Sistema de 5ª vs 1ª Ordem",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude",
    separators=",.",
    template="seaborn",
    legend={"yanchor": "top", "y": 0.99, "xanchor": "left", "x": 0.01},
)

fig.show()

# Exercícios:

Agora que nos familiarizamos com as funções para simular um controlador e gerar o diagrama do lugar das raízes, iniciaremos as atividades do laboratório computacional.

### Atividade 1: Criação do diagrama do lugar das raízes

Considerando o sistema de controle do ângulo de posicionamento de uma antena, descrito no exemplo 2 das notas de aula (https://bernardoschwedersky.github.io/SistemasDeControle/ProjetoLugarDasRaizes.html), e apresentado na figura a seguir:


